In [7]:
import geopandas as gpd
import rasterio
from rasterio import features
from shapely.geometry import shape
import os
import pandas as pd

# Paths
base_path = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data"
shapefile_names = [
    "01_AOI_Achensee",
    "02_AOI_Kuehtai",
    "03_AOI_Kaunertal"
]
shapefile_paths = [os.path.join(base_path, "shapefiles", name + ".shp") for name in shapefile_names]
raster_mask_path = os.path.join(base_path, "ses_topo_22_forest_mask_domain_epsg32632.tif")
seen_shp_path = os.path.join(base_path, "shapefiles", "Seen_AOIs.shp")
gletscher_shp_path_oetztal = os.path.join(base_path, "shapefiles", "GI_5", "Oetztaler_Alpen_GI5.shp")
gletscher_shp_path_stubai = os.path.join(base_path, "shapefiles", "GI_5", "Stubai_GI5.shp")
output_dir = os.path.join(base_path, "shapefiles", "AOIs_masked")
os.makedirs(output_dir, exist_ok=True)

# Load mask shapefiles
seen_gdf = gpd.read_file(seen_shp_path)
gletscher_gdf_oetztal = gpd.read_file(gletscher_shp_path_oetztal)
gletscher_gdf_stubai = gpd.read_file(gletscher_shp_path_stubai)

# Check and unify CRS
if gletscher_gdf_oetztal.crs != gletscher_gdf_stubai.crs:
    gletscher_gdf_stubai = gletscher_gdf_stubai.to_crs(gletscher_gdf_oetztal.crs)
    
gletscher_gdf = pd.concat([gletscher_gdf_oetztal, gletscher_gdf_stubai], ignore_index=True)

# Reproject everything to the CRS of the AOI shapefiles
# Load one AOI to get the target CRS
sample_gdf = gpd.read_file(shapefile_paths[0])
target_crs = sample_gdf.crs

seen_gdf = seen_gdf.to_crs(target_crs)
gletscher_gdf = gletscher_gdf.to_crs(target_crs)

# Load raster mask as polygons (keep areas where mask == 0)
with rasterio.open(raster_mask_path) as src:
    mask_array = src.read(1)
    mask_shapes = [
        shape(geom)
        for geom, val in features.shapes(mask_array, mask=mask_array == 0, transform=src.transform)
        if val == 0
    ]
    raster_mask_gdf = gpd.GeoDataFrame(geometry=mask_shapes, crs=src.crs)

# Apply masks and save
for shp_path, name in zip(shapefile_paths, shapefile_names):
    gdf = gpd.read_file(shp_path)
    # Intersect with raster mask
    gdf = gpd.overlay(gdf, raster_mask_gdf, how='intersection')
    # Remove lakes
    gdf = gpd.overlay(gdf, seen_gdf, how='difference')
    # Remove glaciers
    gdf = gpd.overlay(gdf, gletscher_gdf, how='difference')
    # Save
    out_path = os.path.join(output_dir, f"{name}_masked.shp")
    gdf.to_file(out_path)

C:\Users\Moritz\AppData\Local\Temp\ipykernel_10904\2280882949.py:56: UserWarning: `keep_geom_type=True` in overlay resulted in 58 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  gdf = gpd.overlay(gdf, raster_mask_gdf, how='intersection')
c:\Users\Moritz\miniforge3\envs\Masterarbeit\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'GroÃregio' to 'GroÃreg'
  ogr_write(
C:\Users\Moritz\AppData\Local\Temp\ipykernel_10904\2280882949.py:56: UserWarning: `keep_geom_type=True` in overlay resulted in 11 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  gdf = gpd.overlay(gdf, raster_mask_gdf, how='intersection')
C:\Users\Moritz\AppData\Local\Temp\ipykernel_10904\2280882949.py:56: UserWarning: `keep_geom_type=True` in overlay resulted in 21 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` 